In [1]:
import json
import os

import torch
from torch import nn
from torch.utils.data import DataLoader
from torch.utils.data import Subset
from tqdm.asyncio import tqdm

from internal.data.mil_dataset import MILDatasetMemmapRanges
from internal.data.shuffle_and_cap_bag import ShuffleAndCapBag
from internal.nn.attention_mil import AttentionMIL

cuda_is_available = False
device = torch.device("cpu")

config = json.load(
    open(os.path.join("processed", "config.json"), "r")
)
OUT_DIR = config["OUT_DIR"]
X_PATH = config["X_PATH"]
M_PATH = config["M_PATH"]
Y_PATH = config["Y_PATH"]
IDX_PATH = config["IDX_PATH"]
LOG_PATH = config["LOG_PATH"]
PATCH_SIZE = config["PATCH_SIZE"]
N_PATCHES = config["N_PATCHES"]
MARGIN = config["MARGIN"]
MASK_PATCH_FRAC = config["MASK_PATCH_FRAC"]
MIN_MASK_PIXELS_SLIDE = config["MIN_MASK_PIXELS_SLIDE"]
MIN_MASK_IN_PATCH = config["MIN_MASK_IN_PATCH"]
MIN_TISSUE_FRAC = config["MIN_TISSUE_FRAC"]
MIN_PATCH_PER_SLIDE = config["MIN_PATCH_PER_SLIDE"]
MIN_CENTER_DIST = config["MIN_CENTER_DIST"]
MAX_TRIES_PER_SLIDE = config["MAX_TRIES_PER_SLIDE"]
MAX_TRIES_PER_PATCH = config["MAX_TRIES_PER_PATCH"]
CLASS2ID = config["CLASS2ID"]
ID2CLASS = config["ID2CLASS"]

RETURN_META = True

In [2]:
def mil_collate(batch):
    # batch items can be (x,y) or (x,y,meta)
    if len(batch[0]) == 3:
        xs, ys, metas = zip(*batch)
    else:
        xs, ys = zip(*batch)
        metas = None

    ys = torch.tensor(ys, dtype=torch.long)
    bag_sizes = torch.tensor([x.size(0) for x in xs], dtype=torch.long)
    xcat = torch.cat(xs, dim=0)  # (sum n_i, C, H, W)

    if metas is None:
        return xcat, ys, bag_sizes
    return xcat, ys, bag_sizes, metas


def mil_collate_with_meta(batch):
    xs = [b[0] for b in batch]
    ys = torch.stack([b[1] if torch.is_tensor(b[1]) else torch.tensor(b[1], dtype=torch.long) for b in batch]).long()
    metas = [b[2] for b in batch]  # list of dicts

    bag_sizes = torch.tensor([x.size(0) for x in xs], dtype=torch.long)
    xcat = torch.cat(xs, dim=0)

    return xcat, ys, bag_sizes, metas

def mil_collate_concat(batch):
    if RETURN_META:
        xs, ys, _ = zip(*batch)   # each x: (n_i,C,H,W)
    else:
        xs, ys = zip(*batch)      # each x: (n_i,C,H,W)
    bag_sizes = torch.tensor([x.shape[0] for x in xs], dtype=torch.long)
    x = torch.cat(xs, dim=0)  # (sum n_i, C,H,W)
    y = torch.stack(ys)       # (B,)
    return x, y, bag_sizes

In [3]:
train_ds = MILDatasetMemmapRanges(
    x_path=X_PATH,
    y_path=Y_PATH,
    idx_path=IDX_PATH,
    m_path=M_PATH,
    patch_size=PATCH_SIZE,
    bag_transform=ShuffleAndCapBag(max_instances=16),
    return_meta=True
)
val_ds   = MILDatasetMemmapRanges(
    x_path=X_PATH,
    y_path=Y_PATH,
    idx_path=IDX_PATH,
    m_path=M_PATH,
    patch_size=PATCH_SIZE,
    bag_transform=ShuffleAndCapBag(max_instances=16),
    return_meta=True
)
train_loader = DataLoader(
    train_ds,
    batch_size=4,              # number of slides per batch
    shuffle=True,
    num_workers=os.cpu_count() // 2,
    pin_memory=cuda_is_available,
    collate_fn=mil_collate_concat,   # or mil_collate_list
)

# Sanity: one batch
rgb, msk, label = train_ds[0]
print("Train dataset sample (rgb, msk, label):")
print(rgb.shape, msk.shape, label)

rgb, msk, label = val_ds[0]
print("Val dataset sample (rgb, msk, label):")
print(rgb.shape, msk.shape, label)

x, y, bag_sizes = next(iter(train_loader))
print("Train loader batch (x, y, bag_sizes):")
print(x.shape, y.shape, bag_sizes.shape)

Train dataset sample (rgb, msk, label):
torch.Size([9, 4, 384, 384]) torch.Size([]) {'slide_index': 0, 'start': 0, 'end': 9, 'bag_size': 9}
Val dataset sample (rgb, msk, label):
torch.Size([9, 4, 384, 384]) torch.Size([]) {'slide_index': 0, 'start': 0, 'end': 9, 'bag_size': 9}
Train loader batch (x, y, bag_sizes):
torch.Size([35, 4, 384, 384]) torch.Size([4]) torch.Size([4])


In [4]:
# take 4 slides only
small_ds = torch.utils.data.Subset(train_ds, [0,1,2,3])
small_loader = DataLoader(
    small_ds,
    batch_size=4,
    collate_fn=mil_collate,
    shuffle=True,
    pin_memory=cuda_is_available
)

model = AttentionMIL(n_classes=4).to(device)
opt = torch.optim.AdamW(model.parameters(), lr=1e-4)
loss_fn = nn.CrossEntropyLoss()

for step in tqdm(range(200), desc="training health check"):
    batch = next(iter(small_loader))
    if len(batch) == 4:
        xcat, y, bag_sizes, meta = batch
    else:
        xcat, y, bag_sizes = batch
    xcat, y, bag_sizes = xcat.to(device), y.to(device), bag_sizes.to(device)
    # if cuda_is_available:
        # xcat, y, bag_sizes = xcat.cuda(), y.cuda(), bag_sizes.cuda()

    logits = model(xcat, bag_sizes)
    y = y.to(device)
    loss = loss_fn(logits, y)

    opt.zero_grad()
    loss.backward()
    opt.step()

    if step % 20 == 0:
        print(step, loss.item())


training health check:   0%|          | 1/200 [00:08<26:57,  8.13s/it]

0 1.3543215990066528


training health check:  10%|█         | 21/200 [02:14<16:09,  5.42s/it]

20 0.14971977472305298


training health check:  20%|██        | 41/200 [03:57<13:35,  5.13s/it]

...

training health check:  80%|████████  | 161/200 [14:14<03:18,  5.09s/it]

160 0.00038318903534673154


training health check:  90%|█████████ | 181/200 [15:57<01:36,  5.05s/it]

180 0.0003462936438154429


training health check: 100%|██████████| 200/200 [17:36<00:00,  5.28s/it]


In [4]:
class RandomLabelWrapper(torch.utils.data.Dataset):
    def __init__(self, base_ds, n_classes: int, seed: int = 0):
        self.base = base_ds
        self.n_classes = n_classes
        g = torch.Generator().manual_seed(seed)
        self.rand_y = torch.randint(0, n_classes, (len(base_ds),), generator=g)

    def __len__(self):
        return len(self.base)

    def __getitem__(self, idx):
        out = self.base[idx]
        # base returns (x,y) or (x,y,meta)
        if len(out) == 2:
            x, _y = out
            return x, int(self.rand_y[idx].item())
        else:
            x, _y, meta = out
            return x, int(self.rand_y[idx].item()), meta


def mil_collate_tuple(batch):
    # batch items: (x, y) or (x, y, meta)
    xs = [b[0] for b in batch]
    ys = torch.tensor([b[1] for b in batch], dtype=torch.long)
    bag_sizes = torch.tensor([x.size(0) for x in xs], dtype=torch.long)
    xcat = torch.cat(xs, dim=0)  # (sum n_i, C, H, W)

    if len(batch[0]) == 3:
        metas = [b[2] for b in batch]
        return xcat, ys, bag_sizes, metas
    return xcat, ys, bag_sizes

cuda_is_available = False
device = torch.device("cpu")

K = 8  # small number of slides
idxs = list(range(K))
train_small = Subset(train_ds, idxs)

# wrap with random labels
N_CLASSES = 4
train_small_rand = RandomLabelWrapper(train_small, n_classes=N_CLASSES, seed=123)

small_loader = DataLoader(
    train_small_rand,
    batch_size=4,
    shuffle=True,
    num_workers=0,
    pin_memory=cuda_is_available,
    collate_fn=mil_collate_tuple,
)

model = AttentionMIL(n_classes=N_CLASSES).to(device)
opt = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
loss_fn = nn.CrossEntropyLoss()

model.train()
for step in tqdm(range(200), desc="training with random labels"):
    batch = next(iter(small_loader))
    if len(batch) == 4:
        xcat, y, bag_sizes, meta = batch
    else:
        xcat, y, bag_sizes = batch
    xcat, y, bag_sizes = xcat.to(device), y.to(device), bag_sizes.to(device)

    opt.zero_grad(set_to_none=True)
    logits = model(xcat, bag_sizes)
    loss = loss_fn(logits, y)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    opt.step()

    if step % 20 == 0:
        with torch.no_grad():
            pred = logits.argmax(dim=1)
            acc = (pred == y).float().mean().item()
        print(step, float(loss.item()), acc)


training with random labels:   0%|          | 1/200 [00:07<26:24,  7.96s/it]

0 1.3929870128631592 0.0


training with random labels:  10%|█         | 21/200 [02:11<17:16,  5.79s/it]

20 0.3668363690376282 1.0


training with random labels:  20%|██        | 41/200 [04:07<14:55,  5.63s/it]

...

training with random labels:  80%|████████  | 161/200 [15:48<03:53,  5.99s/it]

160 0.00016748227062635124 1.0


training with random labels:  90%|█████████ | 181/200 [17:45<01:49,  5.77s/it]

180 0.0013772654347121716 1.0


training with random labels: 100%|██████████| 200/200 [19:40<00:00,  5.90s/it]
